Problem 1. k-Nearest Neighbors

We will predict Q’s outcome using k-NN with ordinary (Euclidean) distance.
(a) Using the raw (not scaled in any way) feature values, compute the distance from Q to each of the six 
plants. Then classify Q using k = 1 and using k = 3. Do the two values of k agree? Show your distances.

convert table to dataframe
append row q:
append column: distance from Q given calculation

In [1]:
import pandas as pd

data = {
    'Plant': ['P1','P2','P3','P4','P5','P6'],
    'Sunlight (hours/day)': [3,6,7,2,4,5],
    'Water (mL/week)': [410,550,250,200,190,500],
    'Survived (0N/1Y)': [0,1,1,0,0,1]
}

chart = pd.DataFrame(data)

chart



,Plant,Sunlight (hours/day),Water (mL/week),Survived (0N/1Y)
0,P1,3,410,0
1,P2,6,550,1
2,P3,7,250,1
3,P4,2,200,0
4,P5,4,190,0
5,P6,5,500,1


In [2]:
new_row = ['Q',6,400,'?']

chart.loc[len(chart)] = new_row

chart

,Plant,Sunlight (hours/day),Water (mL/week),Survived (0N/1Y)
0,P1,3,410,0
1,P2,6,550,1
2,P3,7,250,1
3,P4,2,200,0
4,P5,4,190,0
5,P6,5,500,1
6,Q,6,400,?


In [3]:
q_water = int(chart.loc[chart['Plant']=='Q','Water (mL/week)'].values[0])
q_sunlight = int(chart.loc[chart['Plant']=='Q', 'Sunlight (hours/day)'].values[0])

chart['distance'] = ((((chart['Sunlight (hours/day)']-(q_sunlight))**2)\
                    + ((chart['Water (mL/week)']-q_water)**2))**0.5).round(4)

In [4]:
chart #need to review the chart, probably hand calculate

,Plant,Sunlight (hours/day),Water (mL/week),Survived (0N/1Y),distance
0,P1,3,410,0,10.4403
1,P2,6,550,1,150.0000
2,P3,7,250,1,150.0033
3,P4,2,200,0,200.0400
4,P5,4,190,0,210.0095
5,P6,5,500,1,100.0050
6,Q,6,400,?,0.0000


Classifications given K=1 and K=3:

K=1 
Nearest neighbor classification = P1:0
Q Classficiation = 0, Did not Survive

K=3
Nearest neighbors classficiation  = [P1:0, P6:1, P2:1]
Q Classification = 1, Did Survive

The two values do not agree. When using K = 1 it's nearest neighbor did not survive, but when K = 3, even though the data point nearest neighbor did not survive, the majority of neighbors did survive. Interestingly, even if Q was only Subtly different that P1, say sunlight the same but 410.001 mL/week of water, it would still produce the opposite result of P1 when k=3. I attribute this to the dataset being so small and not enough information present to give valid conclusions.

(b) Look at what is driving those distances. Which feature is doing almost all of the work, and why? What 
does that mean for which feature k-NN is effectively paying attention to here?

In [5]:
q_water = int(chart.loc[chart['Plant']=='Q','Water (mL/week)'].values[0])
q_sunlight = int(chart.loc[chart['Plant']=='Q', 'Sunlight (hours/day)'].values[0])

chart['distance_sunlight'] = chart['Sunlight (hours/day)']-q_sunlight
chart['distance_water'] = chart['Water (mL/week)']-q_water

chart

,Plant,Sunlight (hours/day),Water (mL/week),Survived (0N/1Y),distance,distance_sunlight,distance_water
0,P1,3,410,0,10.4403,-3,10
1,P2,6,550,1,150.0000,0,150
2,P3,7,250,1,150.0033,1,-150
3,P4,2,200,0,200.0400,-4,-200
4,P5,4,190,0,210.0095,-2,-210
5,P6,5,500,1,100.0050,-1,100
6,Q,6,400,?,0.0000,0,0


These distances are clearly driven by the value of 'Water (mL/week)'. In the Sunlight column we can see relatively extreme changes in magnitude with differences between the input feature and the dataset features being to the magnitude of 2 or 3 times in some instances while the greatest shift in the Water Column only has differences of .3x or so. Intuitively one may think that greater magnitude shifts would have greater effects, but beause KNN uses the formula for distance: The column with the greatest shift in numerical distance will have the greatest effect of the prediction of the algorithm, not necessarily the column with greatest magnitude changes.

(c) Now standardize each feature (subtract the mean, divide by the standard deviation) and recompute 
the k = 1 prediction.  
To save you some calculations: please use these values:  
● sunlight has mean 4.5 and standard deviation ≈ 1.7;  
● water has mean 350 and standard deviation ≈ 144.  
(As a worked example, Q itself standardizes to sunlight (6 − 4.5) / 1.7 ≈ 0.88 and water (400 − 350) / 144 
≈ 0.35; standardize the six plants the same way, then recompute distances.) Does the prediction change? 
Explain what standardizing did and why it matters for this dataset in particular.

In [6]:
chart = chart.drop(columns=['distance_sunlight','distance_water','distance'])
chart

,Plant,Sunlight (hours/day),Water (mL/week),Survived (0N/1Y)
0,P1,3,410,0
1,P2,6,550,1
2,P3,7,250,1
3,P4,2,200,0
4,P5,4,190,0
5,P6,5,500,1
6,Q,6,400,?


In [7]:
chart['sunlight_mean']=4.5
chart['sunlight_std']=1.7
chart['water_mean']=350
chart['water_std']=144

chart

,Plant,Sunlight (hours/day),Water (mL/week),Survived (0N/1Y),sunlight_mean,sunlight_std,water_mean,water_std
0,P1,3,410,0,4.5,1.7,350,144
1,P2,6,550,1,4.5,1.7,350,144
2,P3,7,250,1,4.5,1.7,350,144
3,P4,2,200,0,4.5,1.7,350,144
4,P5,4,190,0,4.5,1.7,350,144
5,P6,5,500,1,4.5,1.7,350,144
6,Q,6,400,?,4.5,1.7,350,144


In [8]:
chart['standardize_sunlight'] = ((chart['Sunlight (hours/day)'] - chart['sunlight_mean']) / chart['sunlight_std']).round(2)
chart['standardize_water'] = ((chart['Water (mL/week)'] - chart['water_mean']) / chart['water_std']).round(2)
chart

,Plant,Sunlight (hours/day),Water (mL/week),Survived (0N/1Y),sunlight_mean,sunlight_std,water_mean,water_std,standardize_sunlight,standardize_water
0,P1,3,410,0,4.5,1.7,350,144,-0.88,0.42
1,P2,6,550,1,4.5,1.7,350,144,0.88,1.39
2,P3,7,250,1,4.5,1.7,350,144,1.47,-0.69
3,P4,2,200,0,4.5,1.7,350,144,-1.47,-1.04
4,P5,4,190,0,4.5,1.7,350,144,-0.29,-1.11
5,P6,5,500,1,4.5,1.7,350,144,0.29,1.04
6,Q,6,400,?,4.5,1.7,350,144,0.88,0.35


In [9]:
q_water = int(chart.loc[chart['Plant']=='Q','Water (mL/week)'].values[0])
q_sunlight = int(chart.loc[chart['Plant']=='Q', 'Sunlight (hours/day)'].values[0])

chart['non_standardized_distance'] = ((((chart['Sunlight (hours/day)']-(q_sunlight))**2)\
                    + ((chart['Water (mL/week)']-q_water)**2))**0.5).round(2)

q_water_standardized = float(chart.loc[chart['Plant']=='Q','standardize_water'].values[0])
q_sunlight_standardized = float(chart.loc[chart['Plant']=='Q', 'standardize_sunlight'].values[0])

chart['standardized_distance'] = ((((chart['standardize_sunlight']-(q_sunlight_standardized))**2)\
                    + ((chart['standardize_water']-q_water_standardized)**2))**0.5).round(2)

chart

,Plant,Sunlight (hours/day),Water (mL/week),Survived (0N/1Y),sunlight_mean,sunlight_std,water_mean,water_std,standardize_sunlight,standardize_water,non_standardized_distance,standardized_distance
0,P1,3,410,0,4.5,1.7,350,144,-0.88,0.42,10.44,1.76
1,P2,6,550,1,4.5,1.7,350,144,0.88,1.39,150.00,1.04
2,P3,7,250,1,4.5,1.7,350,144,1.47,-0.69,150.00,1.20
3,P4,2,200,0,4.5,1.7,350,144,-1.47,-1.04,200.04,2.73
4,P5,4,190,0,4.5,1.7,350,144,-0.29,-1.11,210.01,1.87
5,P6,5,500,1,4.5,1.7,350,144,0.29,1.04,100.00,0.91
6,Q,6,400,?,4.5,1.7,350,144,0.88,0.35,0.00,0.00


In [10]:
q_sunlight_standardized


0.88

In [11]:
chart['standardized_distance'].sort_values()

6    0.00
5    0.91
1    1.04
2    1.20
0    1.76
4    1.87
3    2.73
Name: standardized_distance, dtype: float64

Non Standardized Classifications given K=1 and K=3:

K=1 
Nearest neighbor classification = P1:0
Q Classficiation = 0, Did not Survive

K=3
Nearest neighbors classficiation  = [P1:0, P6:1, P2:1]
Q Classification = 1, Did Survive

Standardized Classifications given K=1 and K=3:

K=1 
Nearest neighbor classification = P6:1
Q Classficiation = 1, Did Survive

K=3
Nearest neighbors classficiation  = [P2:1, P6:1, P3:1]
Q Classification = 1, Did Survive

The prediction did change when applying standardization to the features. Standardization works by creating a common playing field for the shifts in values between features. The first step, subtracting the mean, works to center the data around 0 or turn all values in a features into an expression of how far they are away from average. The second step, dividing by the standard deviation, creates a unit show how far from average the event is given the rest of the data. With these 2 shifts, even huge changes in features are all boiled to how "normal" are these changes in the features given the rest of the points in that feature set. In turn all features in the dataset are brought into that common context.

Problem 2. Judging a classifier 
The garden built a survival model and used it to flag seedlings that are at risk (predicted not to survive) 
so a volunteer can give them extra care.  

Treat “at risk” (flagged / actually died) as the positive class. 

In [12]:
data2= {
    'Seedling': [1,2,3,4,5,6,7,8,9,10],
    'Flagged': [1,1,1,0,0,1,1,0,0,0],
    'Died':[1,1,0,1,0,1,0,0,0,0]
}

chart2= pd.DataFrame(data2)

chart2

,Seedling,Flagged,Died
0,1,1,1
1,2,1,1
2,3,1,0
3,4,0,1
4,5,0,0
5,6,1,1
6,7,1,0
7,8,0,0
8,9,0,0
9,10,0,0


(a) Compute precision, recall, F1-score, and accuracy. Show your work. 

precision = TP / (TP+FP) If i guessed true, how many guesses were correct?

recall = TP / (TP+FN) How many Trues did i correctly guess out of all the opportunities to guess true?

f1 = 2TP / (2TP + FP + FN) Balancing Recall and Precision

Accuracy = (TP + TN) / (TP + TN + FP + FN) How many of my guesses were correct out of all my guesses?


In [13]:
TP = ((chart2['Flagged']==1) & (chart2['Died'] == 1)).astype(int).sum()

FP = ((chart2['Flagged']==1) & (chart2['Died'] == 0)).astype(int).sum()

TN = ((chart2['Flagged']==0) & (chart2['Died'] == 0)).astype(int).sum()

FN = ((chart2['Flagged']==0) & (chart2['Died'] == 1)).astype(int).sum()

print(f'TP:{TP} FP:{FP} TN:{TN} FN:{FN}')


TP:3 FP:2 TN:4 FN:1


In [14]:
precision = (TP / (TP + FP)).round(2)
recall = (TP / (TP + FN)).round(2)
f1 = ((2*TP) / ((2*TP) + FP + FN)).round(2)
accuracy = ((TP + TN) / (TP + TN + FP + FN)).round(2)

print(f'precision: {precision}\nrecall:{recall}\nf1:{f1}\naccuracy:{accuracy}')

precision: 0.6
recall:0.75
f1:0.67
accuracy:0.7


(b) In this setting, which is worse: a false positive (fussing over a seedling that would have been fine) or a 
false negative (missing one that then dies)? Given your answer, which metric would you prioritize when 
tuning the model, and why? 

In this setting a false negative is worse because a false positive result would just mean more care to an already thriving plant which may make it thrive more, a false negative would be a complete miss of an opportunity to have intervened in a case. In this case maybe I would want to look at the recall of the negative outcome. How many times out of all the possible times to guess negative did I correctly guess negative?

In [15]:
negative_recall = (TN / (TN+FN)).round(2)

print(negative_recall)

0.8


(c) The volunteer boasts that an earlier version of the model got 100% accuracy, but it turns out they 
measured that accuracy on the very same plants the model had been trained on. Explain why that is a 
problem and what they should have done instead. 

The earlier model just learned the existing datapoints and repeated the ouput of the exact same data points when it was fed back to it. Instead, they should have split the data into training and testing data. The model should have been trained on 80% of the data tested on the %20, that way the model is actual performing on data it has never seen before. If tuning the model, to be completely unbiased, they should instead do an 80/10/10 split and tune to the first 10% and test to the latter 10%. This allows for a true test of generalization because it is possibly to tune a model to known outcomes and not being able to generalize to new data. Stratification is also very important as well, the distribution of labels should be similar across all datasets. This dataset is fairly small though so an 80/10/10 might not be ideal.

Problem 3. Logistic regression:  
Instead of a hard yes/no, logistic regression predicts a probability of survival. For a plant with feature 
vector x, it computes z = w · x + b and predicts p = σ(z) = 1 / (1 + e^(−z)), with negative log loss  
L = −[ y · ln(p) + (1 − y) · ln(1 − p) ]. 
Training nudges w and b “downhill” on L. The gradients (which we also used in class) are  
∂L/∂w = (p − y) x and ∂L/∂b = (p − y). 
Consider one plant with (standardized) features x = (2, −1), meaning lots of sun and below-average 
water, that truly survived (y = 1). The current parameters are w = (0.5, 1) and b = 0, and the learning rate 
is η = 0.1.

(a) Compute z, then p = σ(z), then the loss L for this plant.

<img src="2239.jpg" width="400" height="500">

(b) Compute the gradient: ∂L/∂w₁, ∂L/∂w₂, and ∂L/∂b using the formulas above. 

<img src="2240.jpg" width="400" height="500">


B should be .05

(c) Take one gradient-descent step: update each weight and the bias with w ← w − η ∂L/∂w (and 
likewise for b). Report the new w and b. 

<img src="bias fixed.jpg" width="400" height="500">



[510 only] 
(d) Add regularization. We will now also penalize large weights, minimizing L + (λ/2)ǁwǁ² with λ = 0.2 
(the bias is not penalized). Work out the derivative of the penalty term (λ/2)ǁwǁ² with respect to w , add 
it to your gradients from (b), and recompute the weight update. Compare the new weights to your 
answer in (c): what is the penalty doing to them? 

<img src="2243.jpg" width="400" height="500">

<img src="2244.jpg" width="400" height="500">

The penalty is minimizing the impact the gradient has on the weights based on how extreme the change in value was. Larger values and larger shifts experience larger penalties. The shift of the inital W1 of .5 to final of .6 incurs a larger penalty of .088 because of the .1 difference while initial W2 of 1 to the final of .95 recieves a smaller correction at .069

Below are six plants in a (standardized) two-feature space; + means survived, − means did not.

(a) Draw it. Plot the six points. Draw a straight line that separates the two classes. An SVM chooses the 
boundary to make the margin around this line as wide as possible. From your drawing, which plants sit 
on the margin, i.e., the support vectors? Name them (use the letters from the “Plant” column), and 
mention in a sentence what makes the SVM’s choice of boundary the “best” one.

<img src="2245.jpg" width="400" height="500">

Plants sitting on Margin:

Plant A (5,4)
Plant D (3,4)

The SVM's boundary choice is the best one because it allows for the largest "street" seperating the two classes, you could not rotate the linear marging or draw it anywhere else that would separate the classes better.

(b) Two losses on two points. The functional margin of a point is m = y · z, where z = w · x + b is the 
model’s score and y is +1 or −1. The hinge loss (used by SVMs) is max(0, 1 − m); the log loss (logistic 
regression, version that works for ±1 labels) is ln(1 + e^(−m)). Compute both losses for each of these two 
points (these are new points, not from the table above):

Point G, where m = 2.:

<img src="2265.jpg" width="400" height="500">

Point H, where m = 0.5.:

<img src="2264.jpg" width="400" height="500">

(c) How are these points, G and H, treated differently by the two loss functions? In general, how does 
this kind of difference change which set of training data points affect the decision boundary when 
updating parameters to minimize the loss? 

In Point G the Hinge Loss is 0 meaning the datapoint falls safely outside of the margin produced by the svm. Since the model performed very well on this particular datapoint the log loss function only assignes a small penalty of .13 

Point H was a different story, the hinge loss calculated to .5 which means the datapoint lies within the margin of the svm. In turn the Log Loss function penalizes heavier with a value of .47 because the model isn't very confident in it's prediction requiring a greater shift

Point G was a very safe classification for the model, well outside the margin while Point H was ambiguous to the model, it lied within the margin and as such was penalized at a higher value.

The hinge loss would only take into account Point H with a value of .5 because it was within the margin meaning the margin would have to be reevaluated. Log Loss on the other hand would take into account both points but H would have a more significant impact because of it's larger Log Loss value.


PEER REVIEW:

Alseem was very helpful during the review. She pointed out some areas where I had issues and I was able to go back and correct my mistakes. We were able to get through every question during the review and were ready for the Full Class Review.
